In [1]:
import torch
import numpy as np

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)

Y:\Python\AI ML\Month 2\week 7-Deep Learning\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import transformers
import datasets

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce GTX 1650


In [3]:
dataset = load_dataset(
    "universal-dependencies/universal_dependencies",
    "en_ewt"
)

In [4]:
dataset

DatasetDict({
    dev: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 2001
    })
    test: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 2077
    })
    train: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes'],
        num_rows: 12544
    })
})

In [5]:
dataset["train"].features

{'sent_id': Value('string'),
 'text': Value('string'),
 'comments': List(Value('string')),
 'tokens': List(Value('string')),
 'lemmas': List(Value('string')),
 'upos': List(Value('string')),
 'xpos': List(Value('string')),
 'feats': List(Value('string')),
 'head': List(Value('int16')),
 'deprel': List(Value('string')),
 'deps': List(Value('string')),
 'misc': List(Value('string')),
 'mwt': List({'id': Value('string'), 'form': Value('string'), 'feats': Value('string'), 'misc': Value('string')}),
 'empty_nodes': List({'id': Value('string'), 'form': Value('string'), 'lemma': Value('string'), 'upos': Value('string'), 'xpos': Value('string'), 'feats': Value('string'), 'head': Value('string'), 'deprel': Value('string'), 'deps': Value('string'), 'misc': Value('string')})}

In [6]:
example = dataset["train"][0]

print(example["tokens"])
print(example["upos"])

['Al', '-', 'Zaman', ':', 'American', 'forces', 'killed', 'Shaikh', 'Abdullah', 'al', '-', 'Ani', ',', 'the', 'preacher', 'at', 'the', 'mosque', 'in', 'the', 'town', 'of', 'Qaim', ',', 'near', 'the', 'Syrian', 'border', '.']
['PROPN', 'PUNCT', 'PROPN', 'PUNCT', 'ADJ', 'NOUN', 'VERB', 'PROPN', 'PROPN', 'PROPN', 'PUNCT', 'PROPN', 'PUNCT', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN', 'ADP', 'DET', 'NOUN', 'ADP', 'PROPN', 'PUNCT', 'ADP', 'DET', 'ADJ', 'NOUN', 'PUNCT']


In [7]:
label_names = sorted(
    set(
        label
        for example in dataset["train"]["upos"]
        for label in example
    )
)

label_names

['ADJ',
 'ADP',
 'ADV',
 'AUX',
 'CCONJ',
 'DET',
 'INTJ',
 'NOUN',
 'NUM',
 'PART',
 'PRON',
 'PROPN',
 'PUNCT',
 'SCONJ',
 'SYM',
 'VERB',
 'X']

In [8]:
label2id = {
    label: idx
    for idx, label in enumerate(label_names)
}

id2label = {
    idx: label
    for idx, label in enumerate(label_names)
}

In [9]:
print(label2id)
print()
print(id2label)

{'ADJ': 0, 'ADP': 1, 'ADV': 2, 'AUX': 3, 'CCONJ': 4, 'DET': 5, 'INTJ': 6, 'NOUN': 7, 'NUM': 8, 'PART': 9, 'PRON': 10, 'PROPN': 11, 'PUNCT': 12, 'SCONJ': 13, 'SYM': 14, 'VERB': 15, 'X': 16}

{0: 'ADJ', 1: 'ADP', 2: 'ADV', 3: 'AUX', 4: 'CCONJ', 5: 'DET', 6: 'INTJ', 7: 'NOUN', 8: 'NUM', 9: 'PART', 10: 'PRON', 11: 'PROPN', 12: 'PUNCT', 13: 'SCONJ', 14: 'SYM', 15: 'VERB', 16: 'X'}


In [10]:
model_checkpoint = "distilbert/distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(
    model_checkpoint
)

In [11]:
# Implement tokenization + alignment

def tokenize_and_align_pos_labels(examples):

    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True,
    )

    all_labels = []

    for i, pos_tags in enumerate(examples["upos"]):

        word_ids = tokenized_inputs.word_ids(
            batch_index=i
        )

        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:

            # Special tokens such as [CLS] and [SEP]
            if word_idx is None:
                label_ids.append(-100)

            # First subword of an original word
            elif word_idx != previous_word_idx:

                pos_tag = pos_tags[word_idx]

                label_ids.append(
                    label2id[pos_tag]
                )

            # Remaining subwords
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels

    return tokenized_inputs

In [12]:
tokenized_dataset = dataset.map(
    tokenize_and_align_pos_labels,
    batched=True,
)

Map: 100%|██████████| 12544/12544 [00:00<00:00, 16441.98 examples/s]


In [13]:
tokenized_dataset

DatasetDict({
    dev: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2001
    })
    test: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 2077
    })
    train: Dataset({
        features: ['sent_id', 'text', 'comments', 'tokens', 'lemmas', 'upos', 'xpos', 'feats', 'head', 'deprel', 'deps', 'misc', 'mwt', 'empty_nodes', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 12544
    })
})

In [14]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

In [15]:
model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1284.61it/s]
[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
print(model.config.num_labels)
print(model.config.id2label)

17
{0: 'ADJ', 1: 'ADP', 2: 'ADV', 3: 'AUX', 4: 'CCONJ', 5: 'DET', 6: 'INTJ', 7: 'NOUN', 8: 'NUM', 9: 'PART', 10: 'PRON', 11: 'PROPN', 12: 'PUNCT', 13: 'SCONJ', 14: 'SYM', 15: 'VERB', 16: 'X'}


In [17]:
def compute_metrics(eval_predictions):

    predictions, labels = eval_predictions

    predictions = np.argmax(
        predictions,
        axis=2
    )

    valid_predictions = []
    valid_labels = []

    for prediction, label in zip(
        predictions,
        labels
    ):

        for pred, true_label in zip(
            prediction,
            label
        ):

            if true_label != -100:

                valid_predictions.append(pred)
                valid_labels.append(true_label)

    valid_predictions = np.array(
        valid_predictions
    )

    valid_labels = np.array(
        valid_labels
    )

    accuracy = (
        valid_predictions == valid_labels
    ).mean()

    return {
        "accuracy": accuracy
    }

In [18]:
training_args = TrainingArguments(
    output_dir="./pos-distilbert",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=3,

    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,

    report_to="none",

    seed=42,
)

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["dev"],

    processing_class=tokenizer,

    data_collator=data_collator,

    compute_metrics=compute_metrics,
)

In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.415152,0.132560,0.964649
2,0.076418,0.124844,0.967433
3,0.056938,0.123453,0.968785


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.84it/s]


TrainOutput(global_step=2352, training_loss=0.146863106156693, metrics={'train_runtime': 382.6655, 'train_samples_per_second': 98.342, 'train_steps_per_second': 6.146, 'total_flos': 500443158779232.0, 'train_loss': 0.146863106156693, 'epoch': 3.0})

In [21]:
results = trainer.evaluate()

results

Training Loss,Validation Loss,Epoch,Accuracy
0.056938,0.123453,3,0.968785


{'eval_loss': 0.12345252186059952, 'eval_accuracy': 0.9687847940194051}

In [22]:
from transformers import pipeline

pos_pipeline = pipeline(
    "ner",
    model=model,
    tokenizer=tokenizer,
    aggregation_strategy="simple"
)

In [23]:
text = "Apple CEO Tim Cook visited New York last Monday."

entities = pos_pipeline(text)

entities

[{'entity_group': 'PROPN',
  'score': np.float32(0.99791986),
  'word': 'apple',
  'start': 0,
  'end': 5},
 {'entity_group': 'NOUN',
  'score': np.float32(0.96288335),
  'word': 'ceo',
  'start': 6,
  'end': 9},
 {'entity_group': 'PROPN',
  'score': np.float32(0.9980011),
  'word': 'tim cook',
  'start': 10,
  'end': 18},
 {'entity_group': 'VERB',
  'score': np.float32(0.99809307),
  'word': 'visited',
  'start': 19,
  'end': 26},
 {'entity_group': 'ADJ',
  'score': np.float32(0.9969484),
  'word': 'new',
  'start': 27,
  'end': 30},
 {'entity_group': 'PROPN',
  'score': np.float32(0.99826044),
  'word': 'york',
  'start': 31,
  'end': 35},
 {'entity_group': 'ADJ',
  'score': np.float32(0.99534434),
  'word': 'last',
  'start': 36,
  'end': 40},
 {'entity_group': 'PROPN',
  'score': np.float32(0.9968346),
  'word': 'monday',
  'start': 41,
  'end': 47},
 {'entity_group': 'PUNCT',
  'score': np.float32(0.99919444),
  'word': '.',
  'start': 47,
  'end': 48}]